In [3]:
import geopandas as gpd
import pandas as pd
import folium

# Cálculo do PNB de São Paulo

## PNB: ciclovias atuais

In [4]:
from shapely.ops import unary_union
from shapely import MultiLineString

In [ ]:
# creating the bikepath polygon - represents 300m of every bikepath on the city

# get all bikepaths
bikepaths_GDF = gpd.read_file('./data/Ciclorrotas.shp')
bikelanes_GDF = gpd.read_file('./data/Ciclovias.shp')
today_bikelanes_GDF = pd.concat([bikepaths_GDF, bikelanes_GDF])

# clear gdfs from multilinestrings, they stop later processes
multilinestring_rows = today_bikelanes_GDF.geometry.apply(lambda geom: isinstance(geom, MultiLineString))
today_bikelanes_GDF = today_bikelanes_GDF[~multilinestring_rows]
today_bikelanes_GDF = today_bikelanes_GDF.reset_index()

# switch crs modes, create 300m buffer around all bikepaths
today_bikelanes_GDF = today_bikelanes_GDF.set_crs(epsg=4674)
today_bikelanes_GDF = today_bikelanes_GDF.to_crs(epsg=31983) # web mercator, 1:1 com metros
today_bikelanes_GDF_bufferzone = gpd.GeoDataFrame(geometry=[])
today_bikelanes_GDF_bufferzone['geometry'] = today_bikelanes_GDF.geometry.buffer(300)
today_bikelanes_GDF_bufferzone = today_bikelanes_GDF_bufferzone.set_crs(epsg=31983)
today_bikelanes_GDF_bufferzone = today_bikelanes_GDF_bufferzone.to_crs(epsg=4674)

# make buffer polygons into one single entity
today_bikelanes_merged_polygon = unary_union(today_bikelanes_GDF_bufferzone.geometry)

# Experimentos: Observando distorção na leitura de caminhos teorizados

## Teste 1: comparando a distância do graphhopper com a do geopandas
### Resultado:
Não há indício de falta de precisão, os caminhos não foram bem traduzidos com o que eu tinha em mente, acabei pegando
uma região sem cobertura da API, entretanto os resultados do Teste 2 demonstram que a origem do problema era outra.

In [ ]:
# import requests
# import json

# km1_route = ((-46.51434757906752, -23.470215175672507), (-46.51435841408025, -23.470209625229398))
# km10_route = ((-46.51434757906752, -23.470215175672507), (-46.42491164701668, -23.43466451852947))
# km100_route = ((-42.37912639481385, -21.120442825050468), (-42.179999191405315, -20.445098437427568))


# graph_hopper_key = '9463c059-8653-4efd-be7e-548ef8f697b8'

# url = 'https://graphhopper.com/api/1/route?key=' + graph_hopper_key + \
#     '&point={},{}&point={},{}&vehicle=bike&elevation=true&type=json&points_encoded=false'

# request_km1 = requests.get(url.format(km1_route[0][1], km1_route[0][0], km1_route[1][1], km1_route[1][0]))
# request_km10 = requests.get(url.format(km10_route[0][1], km10_route[0][0], km10_route[1][1], km10_route[1][0]))
# request_km100 = requests.get(url.format(km100_route[0][1], km100_route[0][0], km100_route[1][1], km100_route[1][0]))

# experimental_paths = [request_km1.json(), request_km10.json(), request_km100.json()]

# with open('distance_experiment_paths.json', 'w', encoding='utf-8') as f:
#     json.dump(experimental_paths, f, ensure_ascii=False, indent=4)

# coordinates = request_km10.json()['paths'][0]['snapped_waypoints']['coordinates']
# linestring_coordinates = LineString([(point[0], point[1]) for point in coordinates])

experimental_paths_coordinates = [i['paths'][0]['snapped_waypoints']['coordinates'] for i in experimental_paths]
experimental_paths_distances = [i['paths'][0]['distance'] for i in experimental_paths]
experimental_paths_linestring_coordinates = [LineString([(point[0], point[1]) for point in entry]) for entry in experimental_paths_coordinates]

experimental_paths_dict = {'distances_graphhopper':experimental_paths_distances, 'geometry':experimental_paths_linestring_coordinates}

epc_GDF = gpd.GeoDataFrame(experimental_paths_dict)

epc_GDF = epc_GDF.set_crs(epsg=4326)
epc_GDF = epc_GDF.to_crs(epsg=31983)

for index, row in epc_GDF.iterrows():
    print("\n \n graphhopper distance:" + str(row['distances_graphhopper']) + "\n calculated_distance:" + str(row['geometry'].length))

## Teste 2: Comparando com o afirmado pela CET

#### Conclusão: Não há erro de cálculos. Há erro nas informações disponibilizadas.

### Teste: desmontando caminhos para ver se há redundâncias nos caminhos (linestrings):
#### Experimento 1: compensar erros de projeção, simplificando o mapa e unindo possíveis redundâncias no caminho.
#### Resultado: Mudança irrelevante nas projeções. Ainda há uma diferença de 20-30% na projeção em comparação aos data oficiais.

#### Experimento 2: comparar o comprimento da soma de cada caminho individual à distância oficial.
#### Resultado final: os mapas oficiais tem distorções entre o tamanho afirmado e o calculado

In [ ]:
# This cell prepares the data to be used, run it before all other cells
import geopandas as gpd
import pandas as pd
from shapely.ops import unary_union, linemerge, snap

bikepaths1_GDF = gpd.read_file('.\\data\\bikelanes_updated\\Ciclorrotas.shp')
bikepaths2_GDF = gpd.read_file('.\\data\\bikelanes_updated\\Ciclovias.shp')
bikepaths_GDF = pd.concat([bikepaths1_GDF, bikepaths2_GDF], ignore_index = True)
bikepaths_GDF = bikepaths_GDF.set_crs(epsg=4326)
bikepaths_GDF = bikepaths_GDF.to_crs(epsg=31983)

distances_dic = {'name':[], 'CET_distance':[], 'difference_ratio':[]}
totally_alright_paths = bikepaths_GDF.dissolve(by = 'programa')
totally_alright_paths = totally_alright_paths.reset_index()
unique_bikepaths_GDF = bikepaths_GDF.drop_duplicates(subset = 'programa', keep = 'first')
unique_bikepaths_GDF = unique_bikepaths_GDF.drop(columns='geometry').merge(
    totally_alright_paths[['programa', 'geometry']],
    on = 'programa',
    how = 'left'
)
distances_dic['name'].extend(unique_bikepaths_GDF['programa'])
distances_dic['CET_distance'].extend(unique_bikepaths_GDF['extensao_c'])
distances_dic['difference_ratio'] = [None for _ in distances_dic['name']]
distances_dic['calculated_distance'] = [None for _ in distances_dic['name']]

distances_GDF = gpd.GeoDataFrame(distances_dic, geometry=unique_bikepaths_GDF['geometry'])

for idx, row in distances_GDF.iterrows():
    calculated_distance = bikepaths_GDF[bikepaths_GDF['programa'] == row['name']].length.sum()
    distances_GDF.at[idx, 'calculated_distance'] = calculated_distance
    distances_GDF.at[idx, 'difference_ratio'] = calculated_distance/distances_GDF.at[idx, 'CET_distance']

distances_GDF

C:\Users\João Rahal\AppData\Local\Temp\ipykernel_2524\3841115376.py:31: RuntimeWarning: divide by zero encountered in scalar divide
  distances_GDF.at[idx, 'difference_ratio'] = calculated_distance/distances_GDF.at[idx, 'CET_distance']
C:\Users\João Rahal\AppData\Local\Temp\ipykernel_2524\3841115376.py:31: RuntimeWarning: divide by zero encountered in scalar divide
  distances_GDF.at[idx, 'difference_ratio'] = calculated_distance/distances_GDF.at[idx, 'CET_distance']


,name,CET_distance,difference_ratio,calculated_distance,geometry
0,CICLORROTA BATURITÉ/ DIAMANTE,548,0.999934,547.963783,"MULTILINESTRING ((333604.312 7392265.892, 3336..."
1,CICLORROTA BROOKLIN,5980,1.445729,8645.457813,"MULTILINESTRING ((326780.787 7385516.396, 3267..."
2,CICLORROTA DOM MACARIO,1626,0.999022,1624.409467,"MULTILINESTRING ((335651.186 7387246.396, 3356..."
3,CICLORROTA JARDIM EUROPA,1288,1.270186,1635.999806,"MULTILINESTRING ((328426.759 7391722.702, 3284..."
4,CICLORROTA JARDINS,4558,1.712778,7806.841118,"MULTILINESTRING ((328890.874 7392696.588, 3289..."
...,...,...,...,...,...
446,CICLOFAIXA JOSE ALVES CUNHA LIMA,1404,0.998352,1401.685726,"MULTILINESTRING ((321393.268 7393054.833, 3213..."
447,CICLOFAIXA CAMARGO,276,0.995884,274.863952,"MULTILINESTRING ((325449.114 7391892.011, 3254..."
448,CICLOFAIXA CARIOCA/ CAMUMU,1197,1.000583,1197.698088,"MULTILINESTRING ((337264.43 7388569.579, 33729..."
449,CICLOFAIXA JOAQUINA RAMALHO,4154,0.692753,2877.696103,"MULTILINESTRING ((335727.479 7398280.996, 3357..."


In [ ]:
# EXPERIMENT 1
# tried snapping and linemerging routes, seemingly no relevant effect on precision values
for idx, row in distances_GDF.iterrows():
    # possibly_redundant_paths = unary_union(bikepaths_GDF[bikepaths_GDF['programa'] == row['name']]['geometry'])
    possibly_redundant_paths = unary_union(bikepaths_GDF[bikepaths_GDF['programa'] == row['name']]['geometry'].simplify(5))
    snapped_paths = snap(possibly_redundant_paths, possibly_redundant_paths, tolerance = 10) 
    # cleaned_paths = linemerge(snapped_paths)
    distances_GDF.at[idx, 'calculated_distance'] = snapped_paths.length
    # distances_GDF.at[idx, 'difference_ratio'] = cleaned_paths.length/distances_GDF.at[idx, 'CET_distance']

print('projection:', round(distances_GDF['calculated_distance'].sum()), 
      '\t individual sum:', bikepaths_GDF['extensao_t'].sum(), 
      '\t CET official:', distances_GDF['CET_distance'].sum())

In [7]:
# EXPERIMENT 2
# official bikelane values and individual lanes' lenght sum show meaningful difference
print('individual length sum:', '\t bikelane official length:')
len_calculated = 0
len_total = 0
for idx, row in unique_bikepaths_GDF.iterrows():
    value = bikepaths_GDF[bikepaths_GDF['programa'] == row['programa']]['extensao_t'].sum()
    len_calculated += value
    len_total += row['extensao_c']
    print(value, '\t \t \t', row['extensao_c'])
print('total calculated:', len_calculated, '\t total official:', len_total)

individual length sum: 	 bikelane official length:
548 	 	 	 548
8661 	 	 	 5980
1626 	 	 	 1626
1636 	 	 	 1288
7816 	 	 	 4558
7732 	 	 	 7732
3320 	 	 	 3205
4203 	 	 	 3913
8547 	 	 	 4953
2186 	 	 	 1197
2529 	 	 	 2529
524 	 	 	 524
1616 	 	 	 1616
1444 	 	 	 722
8753 	 	 	 7077
2936 	 	 	 2567
1621 	 	 	 1621
443 	 	 	 443
14788 	 	 	 7405
14779 	 	 	 14779
1003 	 	 	 1003
22458 	 	 	 11550
7459 	 	 	 3775
1829 	 	 	 1829
2059 	 	 	 1620
1377 	 	 	 1377
4015 	 	 	 4015
3840 	 	 	 2173
107 	 	 	 0
2100 	 	 	 2100
562 	 	 	 562
293 	 	 	 293
1122 	 	 	 1122
1428 	 	 	 1428
4352 	 	 	 2178
1865 	 	 	 1865
525 	 	 	 525
14046 	 	 	 7280
3900 	 	 	 1954
2975 	 	 	 2959
1394 	 	 	 1394
4218 	 	 	 3331
1728 	 	 	 1728
4113 	 	 	 4113
784 	 	 	 784
4239 	 	 	 3170
2364 	 	 	 2364
670 	 	 	 670
2204 	 	 	 1098
2500 	 	 	 2500
3434 	 	 	 1717
180 	 	 	 180
8608 	 	 	 4458
1008 	 	 	 1008
1420 	 	 	 1420
2850 	 	 	 1572
1801 	 	 	 905
1559 	 	 	 1559
1295 	 	 	 1295
1446 	 	 	 1446
763 	 	

In [12]:
bikepaths_year_dic = {'year':[], 'bikepaths_ingrtd':[], 'errors_gte_ratio':[], 'general_proportion':[], 'error_proportion':[], 'year_score':[]}

error_ratio = 1.5

bikepaths_GDF['inauguracao'] = bikepaths_GDF['inauguracao'].astype(str)
bikepaths_GDF['year'] = bikepaths_GDF['inauguracao'].str[:4]
year_and_bikelanes = bikepaths_GDF['year'].value_counts().sort_index()

high_errors = distances_GDF[distances_GDF['difference_ratio'] >= error_ratio]['name']
bikepath_errors = bikepaths_GDF[bikepaths_GDF['programa'].isin(high_errors)]
year_and_errors = bikepath_errors['year'].value_counts().sort_index()

bikepaths_year_DF = pd.DataFrame(data = {
    'year': year_and_bikelanes.index,
    'bikepaths_ingrtd': year_and_bikelanes,
    'errors_gte_ratio': year_and_errors.fillna(0)
})

total_bikelanes = bikepaths_year_DF['bikepaths_ingrtd'].sum()
total_errors = bikepaths_year_DF['errors_gte_ratio'].sum()
bikepaths_year_DF['general_proportion'] = bikepaths_year_DF['bikepaths_ingrtd']/total_bikelanes
bikepaths_year_DF['error_proportion'] = bikepaths_year_DF['errors_gte_ratio']/total_errors
bikepaths_year_DF['year_score'] = (1-(bikepaths_year_DF['error_proportion']/bikepaths_year_DF['general_proportion']))*100

display(bikepaths_year_DF.sort_values(by = 'error_proportion', ascending = False))

,year,bikepaths_ingrtd,errors_gte_ratio,general_proportion,error_proportion,year_score
year,,,,,,
2016,2016,382,133.0,0.165440,0.178523,-7.908570
2015,2015,421,121.0,0.182330,0.162416,10.921902
2021,2021,323,120.0,0.139887,0.161074,-15.145345
2020,2020,235,108.0,0.101776,0.144966,-42.437241
2014,2014,421,98.0,0.182330,0.131544,27.854103
2023,2023,141,49.0,0.061065,0.065772,-7.707173
2012,2012,46,43.0,0.019922,0.057718,-189.719872
2024,2024,128,34.0,0.055435,0.045638,17.674077
2017,2017,11,11.0,0.004764,0.014765,-209.932886


## PNB: ciclovias teorizadas no primeiro trabalho

In [ ]:
from shapely import LineString
from shapely.ops import unary_union
import json

file_name = ".//data//caminhos_bike_3000m_11.geojson"
with open(file_name, 'r', encoding='utf-8') as file:
    data = json.load(file)

fmap = folium.Map(location=[-23.5, -46.6], zoom_start=10)

i = 0
new_lanes_GDF = gpd.GeoDataFrame(geometry=[])

while i < len(data['features']):
    coordinates = data['features'][i]['properties']['paths'][0]['points']['coordinates']
    linestring_coordinates = LineString([(point[0], point[1]) for point in coordinates])
    temp_gdf = gpd.GeoDataFrame(geometry=[linestring_coordinates])
    new_lanes_GDF = pd.concat([new_lanes_GDF , temp_gdf], ignore_index=True)
    i += 1

#     folium.GeoJson(
#         data={
#             'type': 'Feature',
#             'geometry': linestring_coordinates.__geo_interface__
#         },
#         name='LineString',
#         style_function=lambda x: {'color': 'red'}
#     ).add_to(fmap)

# fmap

new_lanes_GDF = new_lanes_GDF.set_crs(epsg=4326)
new_lanes_GDF = new_lanes_GDF.to_crs(epsg=31983)

today_bikelanes_multilinestring = unary_union(today_bikelanes_GDF.geometry)
new_lanes_GDF = new_lanes_GDF.geometry.apply(lambda line: line.difference(today_bikelanes_multilinestring))

new_lanes_GDF_bufferzone = gpd.GeoDataFrame(geometry=[])
new_lanes_GDF_bufferzone['geometry'] = new_lanes_GDF.geometry.buffer(300)
new_lanes_GDF_bufferzone = new_lanes_GDF_bufferzone.set_crs(epsg=31983)
new_lanes_GDF_bufferzone = new_lanes_GDF_bufferzone.to_crs(epsg=4326)



theorized_bikelanes_merged_polygon = unary_union(new_lanes_GDF_bufferzone.geometry)



## PNB: Cobertura por zonas

In [22]:
GDF_zones_numbers

,sp_nome,estimate_population,population,PNB_score
0,Central,3.646952e+05,414756.0,0.879301
1,Leste,1.276573e+06,4018071.0,0.317708
2,Norte,5.020737e+05,2208510.0,0.227336
3,Oeste,5.449826e+05,1059572.0,0.514342
4,Sul,9.539037e+05,3566432.0,0.267467


In [21]:
GDF_zones_numbers = GDF_sp_subpref_bufferzones[['sp_nome', 'estimate_population']]
GDF_sp_subpref_zones = gpd.overlay(GDF_saopaulo, sao_paulo_subprefeituras_gdf, keep_geom_type=False, how="intersection")
GDF_sp_subpref_zones = GDF_sp_subpref_zones[~GDF_sp_subpref_zones.duplicated('id')]
GDF_sp_subpref_zones = GDF_sp_subpref_zones.groupby('sp_nome', as_index=False).agg({'populacao': 'sum', 'geometry': 'first'})
GDF_zones_numbers = GDF_zones_numbers.groupby('sp_nome', as_index=False).sum()
GDF_zones_numbers['population'] = GDF_sp_subpref_zones['populacao']
GDF_zones_numbers['PNB_score'] = GDF_zones_numbers['estimate_population'] / GDF_zones_numbers['population'] 


In [ ]:
GDF_saopaulo = gpd.read_file('./data/sao_paulo_demographics.geojson')

sao_paulo_subprefeituras_gdf = gpd.read_file('./data/SIRGAS_SHP_subprefeitura_polygon.shp')
# Given the zones in alphabetical order, the 'Zona' list assign them to the cardinal zone (related by hand)  

# Subprefeitura = sao_paulo_subprefeituras_gdf['sp_nome'].sort_values()

# Subprefeitura = [
#     'ARICANDUVA-FORMOSA-CARRAO', 'BUTANTA', 'CAMPO LIMPO', 'CAPELA DO SOCORRO',
#     'CASA VERDE-CACHOEIRINHA', 'CIDADE ADEMAR', 'CIDADE TIRADENTES', 'ERMELINO MATARAZZO',
#     'FREGUESIA-BRASILANDIA', 'GUAIANASES', 'IPIRANGA', 'ITAIM PAULISTA', 'ITAQUERA',
#     'JABAQUARA', 'JACANA-TREMEMBE', 'LAPA', 'M BOI MIRIM', 'MOOCA', 'PARELHEIROS', 'PENHA',
#     'PERUS', 'PINHEIROS', 'PIRITUBA-JARAGUA', 'SANTANA-TUCURUVI', 'SANTO AMARO',
#     'SAO MATEUS', 'SAO MIGUEL', 'SAPOPEMBA', 'SE', 'VILA MARIA-VILA GUILHERME',
#     'VILA MARIANA', 'VILA PRUDENTE'
# ]

zona = [
    'Leste', 'Oeste', 'Sul', 'Sul',
    'Norte', 'Sul', 'Leste', 'Leste',
    'Norte', 'Leste', 'Sul', 'Leste', 'Leste',
    'Sul', 'Norte', 'Oeste', 'Sul', 'Leste', 'Sul', 'Leste',
    'Norte', 'Oeste', 'Norte', 'Norte', 'Sul',
    'Leste', 'Leste', 'Leste', 'Central', 'Norte',
    'Sul', 'Leste'
]
# making separate polygons for each zone and assigning crs 

sao_paulo_subprefeituras_gdf = sao_paulo_subprefeituras_gdf.sort_values(by='sp_nome')
sao_paulo_subprefeituras_gdf['sp_nome'] = zona
sao_paulo_subprefeituras_gdf = sao_paulo_subprefeituras_gdf.set_crs(epsg=31983)

GDF_sp_subpref_bikelanes = gpd.overlay(sao_paulo_subprefeituras_gdf, today_bikelanes_GDF_bufferzone.to_crs(epsg=31983), keep_geom_type=False, how="intersection")
GDF_sp_subpref_bikelanes = GDF_sp_subpref_bikelanes.dissolve(by='sp_nome', as_index=False)

GDF_sp_subpref_bufferzones = gpd.overlay(GDF_sp_subpref_bikelanes, GDF_saopaulo, keep_geom_type=False, how="intersection")
GDF_saopaulo_filtered = GDF_saopaulo[GDF_saopaulo['id'].isin(GDF_sp_subpref_bufferzones['id'])]
GDF_sp_subpref_bufferzones['area1'] = GDF_sp_subpref_bufferzones.area
GDF_saopaulo_filtered['area2'] = GDF_saopaulo_filtered.area
GDF_merge = GDF_sp_subpref_bufferzones[['id', 'area1']].merge(
    GDF_saopaulo_filtered[['id', 'area2']],)
GDF_sp_subpref_bufferzones['area_ratio'] = GDF_merge['area1']/GDF_merge['area2']
# Por alguma razão, a abordagem GDF_intersection_zones_gdfmerge['areas_ratio'] = GDF_intersection_zones_gdfmerge.to_crs('EPSG:31983').area/GDF_saopaulo_overlaid_zones.to_crs('EPSG:31983').area
# não funciona, mas a abordagem GDF_sp_subpref_bufferzones['area_ratio'] = GDF_merge['area1']/GDF_merge['area2'] funciona perfeitamente
GDF_sp_subpref_bufferzones = GDF_sp_subpref_bufferzones.reset_index(drop=True)
GDF_sp_subpref_bufferzones['estimate_population'] = GDF_sp_subpref_bufferzones['populacao'] * GDF_sp_subpref_bufferzones['area_ratio']
GDF_zones_numbers = GDF_sp_subpref_bufferzones[['sp_nome', 'estimate_population']]
GDF_sp_subpref_zones = gpd.overlay(GDF_saopaulo, sao_paulo_subprefeituras_gdf, keep_geom_type=False, how="intersection")
GDF_sp_subpref_zones = GDF_sp_subpref_zones[~GDF_sp_subpref_zones.duplicated('id')]
GDF_sp_subpref_zones = GDF_sp_subpref_zones.groupby('sp_nome', as_index=False).agg({'populacao': 'sum', 'geometry': 'first'})
GDF_zones_numbers = GDF_zones_numbers.groupby('sp_nome', as_index=False).sum()
GDF_zones_numbers['population'] = GDF_sp_subpref_zones['populacao']
GDF_zones_numbers['PNB_score'] = GDF_zones_numbers['estimate_population'] / GDF_zones_numbers['population'] 
GDF_sp_subpref_bufferzones.to_file('./data/sao_paulo_bikelanes_subprefeituras.geojson', driver='GeoJSON')
GDF_zones_numbers.to_csv('./data/sao_paulo_zones_numbers.csv')

# # matching both gdfs order
# GDF_intersection_zones_gdfmerge = GDF_intersection_zones_gdfmerge.sort_values('id', ascending = True)
# GDF_saopaulo_overlaid_zones = GDF_saopaulo_overlaid_zones.sort_values('id', ascending = True)
# GDF_intersection_zones_gdfmerge = GDF_intersection_zones_gdfmerge.reset_index()
# GDF_saopaulo_overlaid_zones = GDF_saopaulo_overlaid_zones.reset_index()

# # calculating the estimate population in said intersected zones (accounting for parcial intersection)
# GDF_intersection_zones_gdfmerge['areas_ratio'] = GDF_intersection_zones_gdfmerge.to_crs('EPSG:31983').area/GDF_saopaulo_overlaid_zones.to_crs('EPSG:31983').area
# GDF_intersection_zones_gdfmerge['estimate_population'] = GDF_intersection_zones_gdfmerge['populacao'] * GDF_intersection_zones_gdfmerge['areas_ratio']

# GDF_intersection_zones_gdfmerge['estimate_population'].sum()


c:\Users\João Rahal\anaconda3\Lib\site-packages\geopandas\geodataframe.py:1819: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


## PNB: Fundindo polígonos de área

In [14]:
from shapely import union
from shapely.ops import unary_union

today_and_theorized_polygon_merge = union(theorized_bikelanes_merged_polygon, today_bikelanes_merged_polygon)
# theorized_polygon_merge = unary_union(theorized_bikelanes_merged_polygon)


## PNB: Visualização de área PNB

In [7]:
fmap = folium.Map(location=[-23.5, -46.6], zoom_start=10)

In [ ]:
# folium.GeoJson(
#     data={'type': 'Feature', 'geometry': today_bikelanes_merged_polygon.__geo_interface__},
#     style_function=lambda _: {'color': 'red'}
# ).add_to(fmap)

# folium.GeoJson(
#     data={'type': 'Feature', 'geometry': theorized_bikelanes_merged_polygon.__geo_interface__},
#     style_function=lambda _: {'color': 'blue'}
# ).add_to(fmap)

# folium.GeoJson(
#     data={'type': 'Feature', 'geometry': today_and_theorized_polygon_merge.__geo_interface__},
#     style_function=lambda _: {'color': 'red'}
# ).add_to(fmap)

# sao_paulo_subprefeituras_gdf = sao_paulo_subprefeituras_gdf.dissolve(by='sp_nome')
# sao_paulo_subprefeituras_gdf = sao_paulo_subprefeituras_gdf.set_crs(epsg=31983)
# sao_paulo_subprefeituras_gdf = sao_paulo_subprefeituras_gdf.to_crs(epsg=4326)

folium.GeoJson(
    data={'type': 'Feature', 'geometry': sao_paulo_subprefeituras_gdf.__geo_interface__},
    style_function=lambda _: {'color': 'red'}
).add_to(fmap)

In [ ]:
fmap

## PNB: GeoDataFrame de SP com data de população

In [ ]:
GDF_saopaulo = gpd.read_file(".//data//sao_paulo_demographics.geojson")
GDF_saopaulo = GDF_saopaulo.set_crs(epsg=31983,allow_override=True)
GDF_saopaulo = GDF_saopaulo.to_crs(epsg=4326)
# GDF_saopaulo

### Nota:
Eu testei fazer com os cálculos com um método de polígonos multiplos ao invés desse DF com um polígono único, que me pareceu estranho. É menos prático, já que o union() do shapely resulta em uma geoseries ao invés de um GDF. Não tem diferença signigicativa de tempo, então apesar de estranho, o GDF de polígono único foi mais prático. Por fim, quando eu souber mais sobre concorrência e paralelismo posso tentar aproveitar multiplos cores para fazer os calculos de área, que atualmente levam tempo pra rodar. 

In [ ]:
# make the area polygon into a GDF and assign a crs for it 
GDF_today_and_theorized_polygon_merge = gpd.GeoDataFrame(geometry=[today_and_theorized_polygon_merge])
GDF_today_and_theorized_polygon_merge = GDF_today_and_theorized_polygon_merge.set_crs('EPSG:4326')

# finding census zones intersecting the bikelane polygon
GDF_intersection_zones_gdfmerge = gpd.overlay(GDF_saopaulo, GDF_today_and_theorized_polygon_merge, how="intersection")

# removing non-intersecting zones
GDF_saopaulo_overlaid_zones = GDF_saopaulo[GDF_saopaulo['id'].isin(GDF_intersection_zones_gdfmerge['id'])]

# matching both gdfs order
GDF_intersection_zones_gdfmerge = GDF_intersection_zones_gdfmerge.sort_values('id', ascending = True)
GDF_saopaulo_overlaid_zones = GDF_saopaulo_overlaid_zones.sort_values('id', ascending = True)
GDF_intersection_zones_gdfmerge = GDF_intersection_zones_gdfmerge.reset_index()
GDF_saopaulo_overlaid_zones = GDF_saopaulo_overlaid_zones.reset_index()

# calculating the estimate population in said intersected zones (accounting for parcial intersection)
GDF_intersection_zones_gdfmerge['areas_ratio'] = GDF_intersection_zones_gdfmerge.to_crs('EPSG:31983').area/GDF_saopaulo_overlaid_zones.to_crs('EPSG:31983').area
GDF_intersection_zones_gdfmerge['estimate_population'] = GDF_intersection_zones_gdfmerge['populacao'] * GDF_intersection_zones_gdfmerge['areas_ratio']

GDF_intersection_zones_gdfmerge['estimate_population'].sum()

In [ ]:
GDF_intersection_zones_gdfmerge['estimate_population'].sum()

## PHB: Estimando quanto precisa ser construído

In [ ]:
from shapely.ops import unary_union

# from the new lanes, finding how much is coincident to today's lanes
today_bikelanes_multilinestring = unary_union(today_bikelanes_GDF.geometry)
to_be_constructed_lanes_GDF = new_lanes_GDF.geometry.apply(lambda line: line.difference(today_bikelanes_multilinestring))

to_be_constructed_lanes_GDF = to_be_constructed_lanes_GDF.set_crs(epsg=31983)

total_lanes_extension = to_be_constructed_lanes_GDF.geometry.length.sum()
total_lanes_extension

In [ ]:
today_bikelanes_GDF = today_bikelanes_GDF.to_crs(epsg=31983)

today_bikelanes_GDF.geometry.length.sum()

# Estimativa atual inalterada (microdata de 2010):
##   3378449 (30.02%) pessoas à 300m de infraestrutura cicloviária

# Estimativa com rotas teorizadas (1000m 1:1):
##   4320607 (+942158) pessoas à 300m de infraestrutura cicloviária
##   (30.02% -> 38.39%, + 8.37%)
##   necessários 318km de ciclovias novas 

# Estimativa com rotas teorizadas (3000m 1:1):
##   4758654 (+1380205) pessoas à 300m de infraestrutura cicloviária
##   (30.02% -> 42.28%, + 12.26%)
##   necessários 864km de ciclovias novas

# Estimativa com rotas teorizadas (3000m 1:1):
##   5303848 (+1925399) pessoas à 300m de infraestrutura cicloviária
##   (30.02% -> 47.13%, + 17.11%)
##   necessários 1128km de ciclovias novas 

### Nota: Teorizo que esses número podem ser mais baixos. Tentei calcular o comprimento de algumas vias e a extensão total e deu um número significativamente mais inflado que o oficial. 960km de ciclovias em comparação com os 731 calculados pela CET. É bom dar uma investigada

In [ ]:
# 3378449/11253503

# 4320607/11253503
# 38.39-30.02

# 4758654/11253503
# 42.28-30.02

# 5303848/11253503
47.13-30.02

# 2 Parte: Conexão de rotas (feito)

In [ ]:
from shapely.geometry import Point, LineString, MultiLineString
import json
import requests
from time import sleep
import geopandas as gpd
import pandas
import folium
from shapely.ops import unary_union
from collections import Counter

In [ ]:
fmap = folium.Map(location=[-23.5, -46.6], zoom_start=10)


folium.GeoJson(
    data={
        'type': 'Feature',
        'geometry': today_bikelanes_multilinestring.__geo_interface__
    },
    name='LineString',
    style_function=lambda x: {'color': 'red'}
).add_to(fmap)

folium.GeoJson(
    data = loose_ends_gdf,
    name = 'Markers'
).add_to(fmap)


fmap

In [ ]:
bikepaths_GDF = gpd.read_file('C:\\Users\\joao_rahal\\bike-science\\sao-paulo\\bike-infra-maps\\code\\data\\Ciclorrotas.shp')
bikelanes_GDF = gpd.read_file('C:\\Users\\joao_rahal\\bike-science\\sao-paulo\\bike-infra-maps\\code\\data\\Ciclovias.shp')
today_bikelanes_GDF = concat([bikepaths_GDF, bikelanes_GDF])
today_bikelanes_GDF = today_bikelanes_GDF.set_crs('EPSG:4326')

today_bikelanes_multilinestring = unary_union(today_bikelanes_GDF.geometry)

# extract endpoints
def get_endpoints(geom):
    if isinstance(geom, LineString):
        return [geom.coords[0], geom.coords[-1]]
    elif isinstance(geom, MultiLineString):
        return [pt for line in geom.geoms for pt in (line.coords[0], line.coords[-1])]
    return []

endpoints = get_endpoints(today_bikelanes_multilinestring)

# count the number of times endpoint appear, if 1, it is a loose endpoint
endpoints_counter = Counter(endpoints)
loose_ends = [Point(pt) for pt, count in endpoints_counter.items() if count == 1]

loose_ends_gdf = gpd.GeoDataFrame(geometry=loose_ends, crs=today_bikelanes_GDF.crs)

# removing loose ends too close to each other, those are usually bikelanes separated by bridges or double-way bikelanes ending  
loose_ends_gdf = loose_ends_gdf.to_crs(epsg=3857)


loose_ends_gdf

In [ ]:
 = today_bikelanes_GDF.to_crs('EPSG:4326')
# first_points = today_bikelanes_GDF.geometry.apply(lambda line: Point(line.coords[0]) if line else None)
last_points = today_bikelanes_GDF.geometry.apply(lambda line: Point(line.coords[-1]) if line else None)

# first_points_gdf = gpd.GeoDataFrame(gdf.drop(columns="geometry"), geometry=first_points, crs=gdf.crs)
last_points_gdf = gpd.GeoDataFrame(today_bikelanes_GDF.drop(columns="geometry"),geometry=last_points, crs='EPSG:4326')

# today_bikelanes_GDF

def find_closest_pairs(gdf, max_distance):
    if gdf.crs.is_geographic:
        gdf = gdf.to_crs(epsg = 3857)  # CRS métrico

    pairs = []

    for i, point1 in gdf.iterrows():
        distances = gdf.geometry.distance(point1.geometry)
        
        valid_distances = distances[(distances <= max_distance) & (distances > 0)]
        
        if not valid_distances.empty:
            closest_index = valid_distances.idxmin()
            closest_point = gdf.loc[closest_index]
            pairs.append((point1.geometry, closest_point.geometry))
            
            gdf = gdf.drop(closest_index)

    # Step 3: Create the result GeoDataFrame
    geometries_start = [pair[0] for pair in pairs]
    geometries_finish = [pair[1] for pair in pairs]
    # geometries_start = geometries_start.set_crs(epsg = 3857)
    # geometries_finish = geometries_finish.set_crs(epsg = 3857)
    # geometries_start = geometries_start.to_crs(epsg = 4326)
    # geometries_finish = geometries_finish.to_crs(epsg = 4326)
    result_gdf_start = gpd.GeoDataFrame(geometry=geometries_start, crs=gdf.crs)
    result_gdf_finish = gpd.GeoDataFrame(geometry=geometries_finish, crs=gdf.crs)
    result_gdf = gpd.GeoDataFrame(geometry=[])
    result_gdf['polygon1_center'] = result_gdf_start['geometry'] 
    result_gdf['polygon2_center'] = result_gdf_finish['geometry']
    result_gdf['polygon1_center'] = result_gdf['polygon1_center'].to_crs(epsg = 4326)
    result_gdf['polygon2_center'] = result_gdf['polygon2_center'].to_crs(epsg = 4326)
    return result_gdf

# Example usage
GDF_connections = find_closest_pairs(last_points_gdf, 1000)
list(GDF_connections.columns.values)

PATHER - Levemente alterado para funcionar aqui

In [ ]:
import requests
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from time import sleep

def graphhopper_request(url, row):
    """Send a request to the GraphHopper API and return JSON response."""
    coords = (
        row['polygon1_center'].y, row['polygon1_center'].x,
        row['polygon2_center'].y, row['polygon2_center'].x
    )
    
    response = requests.get(url.format(*coords))
    
    if response.status_code == 200:
        return response.json()
    
    raise requests.HTTPError(f"GraphHopper API request failed with status code {response.status_code}")

def convert_list_columns_to_str(df):
    """Convert all list-type columns in a DataFrame to string representation. Needed for dealing with graphhopper requests"""
    for col in df.select_dtypes(include=[object]):  # Optimize column selection
        if df[col].apply(lambda x: isinstance(x, list)).any():
            df[col] = df[col].astype(str)
    return df

def graphhopper_pather(POI_gdf, json_filename, limit):
    """Request paths from the GraphHopper API and save results to a JSON file."""
    graph_hopper_key = '9463c059-8653-4efd-be7e-548ef8f697b8'
    url = (
        f'https://graphhopper.com/api/1/route?key={graph_hopper_key}'
        '&point={},{}&point={},{}&vehicle=bike&elevation=true&type=json&points_encoded=false'
    )

    rows = []
    limit = min(limit, len(POI_gdf))  # Ensure limit does not exceed dataset size

    for count, (_, row) in enumerate(POI_gdf.iterrows()):
        if count >= limit:
            break
        sleep(2)  # Rate limiting, graphhopper server couldn't handle many requests
        rows.append(graphhopper_request(url, row))

    gdf = gpd.GeoDataFrame(rows, crs=POI_gdf.crs)

    if 'geometry' not in gdf.columns:
        gdf['geometry'] = gpd.GeoSeries([Point()] * len(gdf), crs=POI_gdf.crs)

    gdf = convert_list_columns_to_str(gdf)
    
    gdf.to_file(json_filename, driver='GeoJSON')

    return gdf


In [ ]:
geojson_filename = 'sugestoes_de_conexao.json'
suggested_connection_paths = graphhopper_pather(GDF_connections, geojson_filename, 50)

In [ ]:
import json
import folium
import geopandas as gpd
import pandas as pd
from shapely.geometry import LineString

def create_folium_map(geojson_filename, output_html="output.html"):
    """Generate a Folium map from a GeoJSON file and save it as an HTML file."""
    
    fmap = folium.Map(location=[-23.5, -46.6], zoom_start=10)

    with open(geojson_filename, 'r', encoding='utf-8') as file:
        data = json.load(file)

    html_name = geojson_filename.rsplit(".", 1)[0] + ".html"

    coordinates_gdf = gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")

    route_layer = folium.FeatureGroup(name="Suggested Routes", show=True)

    for feature in data.get("features", []):
        coordinates = feature.get("properties", {}).get("paths", [{}])[0].get("points", {}).get("coordinates", [])
        if coordinates:
            linestring = LineString([(point[0], point[1]) for point in coordinates])
            temp_gdf = gpd.GeoDataFrame(geometry=[linestring], crs="EPSG:4326")
            coordinates_gdf = pd.concat([coordinates_gdf, temp_gdf], ignore_index=True)

            folium.GeoJson(
                data={'type': 'Feature', 'geometry': linestring.__geo_interface__},
                name='LineString',
                style_function=lambda _: {'color': 'blue'}
            ).add_to(route_layer)

    route_layer.add_to(fmap)

    if "today_bikelanes_GDF" in globals() and not today_bikelanes_GDF.empty:
        folium.GeoJson(
            data={'type': 'Feature', 'geometry': today_bikelanes_GDF.__geo_interface__},
            name='Historic Paths',
            style_function=lambda _: {'color': 'red'}
        ).add_to(fmap)

    folium.LayerControl().add_to(fmap)

    fmap.save(html_name)
    
    return fmap, coordinates_gdf

geojson_file = "sugestoes_de_conexao.json"
fmap, coordinates_gdf = create_folium_map(geojson_file)
